### Import Dependencies


In [1]:
import openai
import instructor
from pydantic import BaseModel, Field
from qdrant_client import QdrantClient

In [2]:
from dotenv import load_dotenv

load_dotenv('../../.env')

True

### Mock example

In [4]:
prompt = """You are a helpful assistant.
Return an answer to the user's question.
Question: what's your name?
"""

In [6]:
response = openai.chat.completions.create(
    model="gpt-5.4-nano",
    messages=[
        {"role": "system", "content": prompt},
    ],
    reasoning_effort="none"
)

print(response.choices[0].message.content)

I’m ChatGPT.


In [7]:
response

ChatCompletion(id='chatcmpl-DwzJCtGaJ8NhFcfNI0CHJtZiVDUBv', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='I’m ChatGPT.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1782951066, model='gpt-5.4-nano-2026-03-17', object='chat.completion', moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=8, prompt_tokens=26, total_tokens=34, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

### Add Instructor (Structured Outputs)

In [10]:
client = instructor.from_provider(
    "openai/gpt-5.4-nano",
    mode=instructor.Mode.RESPONSES_TOOLS
)

In [11]:
class Answer(BaseModel):
    answer: str = Field(description="The answer to the user's question")

In [12]:
response = client.create(
    messages=[
        {"role": "user", "content": prompt}
    ],
    response_model=Answer,
    reasoning={"effort": "none"}
)

In [13]:
response

Answer(answer='I’m ChatGPT, an AI assistant.')

In [14]:
response = client.create_with_completion(
    messages=[
        {"role": "user", "content": prompt}
    ],
    response_model=Answer,
    reasoning={"effort": "none"}
)

In [15]:
response

(Answer(answer='I’m ChatGPT.'),
 Response(id='resp_03b02fbc8ea925dc006a45aeab3700819e88cec99943a64e92', created_at=1782951595.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-nano-2026-03-17', object='response', output=[ResponseFunctionToolCall(arguments='{"answer":"I’m ChatGPT."}', call_id='call_FHFhMMZNjtC6zab5bn2rDwpD', name='Answer', type='function_call', id='fc_03b02fbc8ea925dc006a45aeab7d3c819ebd97a23ef8041053', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice=ToolChoiceFunction(name='Answer', type='function'), tools=[FunctionTool(name='Answer', parameters={'properties': {'answer': {'description': "The answer to the user's question", 'title': 'Answer', 'type': 'string'}}, 'required': ['answer'], 'title': 'Answer', 'type': 'object', 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Correctly extracted `Answer` with all the required parameters with correct 

In [23]:
class AnswerWithReasoning(BaseModel):
    reasoning: str = Field(description="The reasoning for the answer")
    answer: str = Field(description="The answer to the user's question")   


In [24]:
response, raw_response = client.create_with_completion(
    messages=[
        {"role": "user", "content": prompt}
    ],
    response_model=AnswerWithReasoning,
    reasoning={"effort": "none"}
)

In [25]:
response

AnswerWithReasoning(reasoning="The user asks for my name. As an AI assistant, I should identify myself. I don't have a personal name, but can state that I'm ChatGPT.", answer='I’m ChatGPT.')

### RAG pipeline

In [30]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="The answer to the user's question")   
    

In [37]:
qdrant_client = QdrantClient(url='http://localhost:6333')

def get_embedding(text, model='text-embedding-3-small'):
    response = openai.embeddings.create(
        model=model,
        input=text,
    )
    
    return response.data[0].embedding

def retrieve_data(query, collection_name='amazon-items-collection-01', k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name=collection_name,
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context_scores = []
    retrieved_context_texts = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload['parent_asin'])
        retrieved_context_scores.append(result.score)
        retrieved_context_texts.append(result.payload['processed_description'])
        retrieved_context_ratings.append(result.payload['average_rating'])

    return {
        'retrieved_context_ids': retrieved_context_ids,
        'retrieved_context_scores': retrieved_context_scores,
        'retrieved_context_texts': retrieved_context_texts,
        'retrieved_context_ratings': retrieved_context_ratings
    }

def process_context(retrieve_context):
    formatted_context = ''

    for id, chunk, rating in zip(retrieve_context['retrieved_context_ids'], retrieve_context['retrieved_context_texts'], retrieve_context['retrieved_context_ratings']):
        formatted_context += f"- Product ID: {id}, Product Rating: {rating}, Product Description: {chunk}\n"

    return formatted_context

def build_prompt(question, formatted_context):
    prompt = f"""
    You are a shopping assistant that can answer questions about the products in stock.

    You will be given a question and a list of context.

    Instructions:
    - Answer the question based on the context only.
    - Never use word context and refer to it as the available products.
    - Do not use markdown formatting

    Context:
    {formatted_context}

    Question:
    {question}
    """
    return prompt

def generate_answer(prompt):
    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "user", "content": prompt}
        ],
        response_model=RAGGenerationResponse,
        reasoning={"effort": "none"}
    )

    return response

def rag_pipeline(question, topk=5):
    retrieved_context = retrieve_data(query=question, k=topk)
    formatted_context = process_context(retrieved_context)
    prompt = build_prompt(question, formatted_context)
    answer = generate_answer(prompt)
    
    final_answer = {
        'data_object': answer,
        'answer': answer.answer,
        'question': question,
        'retrieved_context_ids': retrieved_context['retrieved_context_ids'],
        'retrieved_context': retrieved_context['retrieved_context_texts'],
    }

    return final_answer

In [38]:
output = rag_pipeline("Suggest me a laptop", 3)

In [39]:
output

{'data_object': RAGGenerationResponse(answer='If you want a solid everyday laptop, consider the HP 15.6" HD Business Laptop (Product ID: B0C9ZWCZ99). It has an AMD Ryzen 5 5500U (up to 4.0GHz), 8GB RAM, and a 256GB SSD, plus a 15.6" HD micro-edge display and Windows 11 Home. Connectivity includes USB-A and USB-C, and it also has HDMI.'),
 'answer': 'If you want a solid everyday laptop, consider the HP 15.6" HD Business Laptop (Product ID: B0C9ZWCZ99). It has an AMD Ryzen 5 5500U (up to 4.0GHz), 8GB RAM, and a 256GB SSD, plus a 15.6" HD micro-edge display and Windows 11 Home. Connectivity includes USB-A and USB-C, and it also has HDMI.',
 'question': 'Suggest me a laptop',
 'retrieved_context_ids': ['B0C9ZWCZ99', 'B099N9F3FP', 'B09WCT9S1R'],
 'retrieved_context': ['HP 15.6" HD Busienss Laptop Newest, 6-core AMD Ryzen 5 5500U(up to 4.0GHz), 8GB RAM, 256GB SSD, USB-A&C, WiFi, Fast Charge, Windows 11 + GM Accessory 【15.6" HD micro-edge Display】Revolutionize your display and see more of wha